# E6 | Model Clustering K-Means


In [4]:
pip install pandas pyarrow

In [5]:
# Carrega o arquivo Parquet
import pandas as pd
df_parquet = pd.read_parquet('/content/ml_cluster_dataset.parquet')

In [6]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 879.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
import warnings
warnings.filterwarnings('ignore')
import os, pandas as pd, numpy as np, tempfile, joblib
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
import mlflow

load_dotenv()
print('Setup OK')

Setup OK


In [9]:
# ===== [3b] EDA: ANALISE DOS DADOS ANTES DE TRANSFORMACAO =====
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

df = df_parquet.copy() # Adicionei esta linha para definir 'df'

# CAUSA: Features de ENTRADA
cause_cols = [
    'prioridade_num', 'grupo_designado', 'categoria', 'subcategoria', 'produto',
    'hora_abertura', 'turno_abertura', 'dia_semana_num', 'fora_horario_comercial',
    'abriu_fim_de_semana', 'mes_abertura', 'trimestre', 'possui_pai', 'triagem_incompleta'
]

# EFEITO: Features de SAIDA (EXCLUIR do treino)
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# EXCLUIR: NULL placeholders, redundantes
exclude_cols = [
    'incident_id', 'cluster', 'data_abertura',
    'duracao_horas_scaled', 'cluster_id',
    'is_filho_de_problema'
]

print('[STATS] ESTRUTURA DE FEATURES:')
print('   [OK] CAUSA (incluir): %d colunas' % len(cause_cols))
print('   [WARN] EFEITO (interpretar): %d colunas' % len(effect_cols))
print('   [ERROR] EXCLUIR: %d colunas' % len(exclude_cols))
print('   Total dataset: %d' % len(df.columns))

# EDA com 6 graficos
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('EDA: Analise de Features ANTES de Transformacao', fontsize=14, fontweight='bold')

# [1] Cardinalidade
cat_cols_all = df.select_dtypes(include=['object']).columns.tolist()
cardinality = {col: df[col].nunique() for col in cat_cols_all}
cardinality_sorted = dict(sorted(cardinality.items(), key=lambda x: x[1], reverse=True))
axes[0, 0].barh(list(cardinality_sorted.keys()), list(cardinality_sorted.values()), color='steelblue')
axes[0, 0].set_xlabel('Numero de Valores Unicos')
axes[0, 0].set_title('1. Cardinalidade - Colunas Categoricas')
axes[0, 0].grid(axis='x', alpha=0.3)

# [2] Distribuicao
cause_numeric = [c for c in cause_cols if c in df.columns and df[c].dtype in ['int64', 'float64']]
sample_cause = cause_numeric[:4] if len(cause_numeric) >= 4 else cause_numeric
if sample_cause:
    df[sample_cause].hist(ax=axes[0, 1], bins=30, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].set_title('2. Distribuicao - Features CAUSA (amostra)')
axes[0, 1].grid(alpha=0.3)

# [3] Correlacao
if len(cause_numeric) > 1:
    corr_matrix = df[cause_numeric].corr()
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                ax=axes[0, 2], cbar_kws={'label': 'Correlacao'})
axes[0, 2].set_title('3. Correlacao - Features CAUSA')

# [4] NULL values
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]
if len(null_pct) > 0:
    axes[1, 0].barh(null_pct.index, null_pct.values, color='coral')
    axes[1, 0].set_xlabel('Percentual de NULLs')
axes[1, 0].set_title('4. Valores NULL')
axes[1, 0].grid(axis='x', alpha=0.3)

# [5] Categoria distribution
if 'categoria' in df.columns:
    categoria_dist = df['categoria'].value_counts().head(15)
    axes[1, 1].barh(categoria_dist.index, categoria_dist.values, color='mediumseagreen')
    axes[1, 1].set_xlabel('Contagem')
axes[1, 1].set_title('5. Top-15 Categorias')
axes[1, 1].grid(axis='x', alpha=0.3)

# [6] Summary
axes[1, 2].axis('off')
summary = ('RESUMO DE FEATURES:\n\n'
          '[OK] CAUSA: %d colunas\n'
          '     Objetivo: Perfil entrada\n\n'
          '[WARN] EFEITO: %d colunas\n'
          '      Objetivo: Interpretar\n\n'
          '[ERROR] EXCLUIR: %d\n'
          '        Data leakage/NULL') % (len(cause_cols), len(effect_cols), len(exclude_cols))
axes[1, 2].text(0.05, 0.95, summary, transform=axes[1, 2].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1, 2].set_title('6. Estrategia de Feature Selection', fontweight='bold')

plt.tight_layout()
eda_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'eda_feature_selection.png')
Path(eda_path).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(eda_path, dpi=100, bbox_inches='tight')
plt.close()

print('[OK] EDA salva em: %s' % eda_path)

[STATS] ESTRUTURA DE FEATURES:
   [OK] CAUSA (incluir): 14 colunas
   [WARN] EFEITO (interpretar): 7 colunas
   [ERROR] EXCLUIR: 6 colunas
   Total dataset: 26
[OK] EDA salva em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans/eda_feature_selection.png


In [10]:
# ===== [4] FEATURE ENGINEERING: FREQUENCY ENCODING (OTIMIZADO PARA K-MEANS) =====
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

print("\n🔧 FEATURE ENGINEERING: FREQUENCY ENCODING (SEM ONE-HOT ENCODING)")

# 1. Tratamento de Outliers (Lógica original mantida)
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
numeric_cols = [
    c for c in numeric_cols if c not in ["incident_id", "cluster", "data_abertura"]
]

outlier_counts = pd.DataFrame(index=df.index)
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)
  IQR = Q3 - Q1
  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 * IQR
  outlier_counts[col] = ~((df[col] >= lower) & (df[col] <= upper))

outlier_total_per_row = outlier_counts.sum(axis=1)
max_outliers_allowed = len(numeric_cols) * 0.5
outlier_mask = outlier_total_per_row <= max_outliers_allowed

df_clean = df[outlier_mask].copy()

# 2. Seleção de Features de Causa
cause_cols = [
    "prioridade_num",
    "grupo_designado",
    "categoria",
    "subcategoria",
    "produto",
    "hora_abertura",
    "turno_abertura",
    "dia_semana_num",
    "fora_horario_comercial",
    "abriu_fim_de_semana",
    "mes_abertura",
    "trimestre",
    "possui_pai",
    "triagem_incompleta",
]
cause_cols_valid = [c for c in cause_cols if c in df_clean.columns]
df_features = df_clean[cause_cols_valid].copy()

# 3. Encoding Cíclico para Features Temporais
temporal_cyclic = {
    "hora_abertura": 24,
    "dia_semana_num": 7,
    "mes_abertura": 12,
}
for col, period in temporal_cyclic.items():
  if col in df_features.columns:
    df_features[f"{col}_sin"] = np.sin(2 * np.pi * df_features[col] / period)
    df_features[f"{col}_cos"] = np.cos(2 * np.pi * df_features[col] / period)
    df_features = df_features.drop(columns=[col])

# 4. Frequency Encoding para TODAS as categóricas (Evita a explosão de colunas)
all_categorical_cols = [
    "grupo_designado",
    "subcategoria",
    "produto",
    "categoria",
    "turno_abertura",
]
for col in all_categorical_cols:
  if col in df_features.columns:
    freq_map = df_features[col].value_counts(normalize=True).to_dict()
    df_features[col] = df_features[col].map(freq_map).fillna(0)

# 5. Normalização com StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features.fillna(0))

print(
    f"✅ Features Finais (sem One-Hot Encoding): {X_scaled.shape[1]} colunas"
    " numéricas contínuas."
)


🔧 FEATURE ENGINEERING: FREQUENCY ENCODING (SEM ONE-HOT ENCODING)
✅ Features Finais (sem One-Hot Encoding): 17 colunas numéricas contínuas.


In [11]:
# ===== [4b] VISUALIZAÇÃO: IMPACTO DA SELEÇÃO DE FEATURES =====
print('\n📊 VISUALIZAÇÃO: Feature Selection Impact')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Feature Selection Impact: CAUSA + Encoding Otimizado', fontsize=14, fontweight='bold')

# [1] Barplot: Features Antes vs Depois
features_comparison = ['Antes (OHE Explosion)', 'Depois (CAUSA only)']
features_count = [680, X_scaled.shape[1]]
colors_comp = ['#E53935', '#43A047']
bars = axes[0].bar(features_comparison, features_count, color=colors_comp, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Número de Features', fontweight='bold')
axes[0].set_title('1. Redução de Features')
axes[0].grid(axis='y', alpha=0.3)
for bar, count in zip(bars, features_count):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{count}\nfeatures', ha='center', va='bottom', fontweight='bold', fontsize=11)

# [2] Pie Chart: Qual % foi removido e por quê
removed_count = 680 - X_scaled.shape[1]
kept_count = X_scaled.shape[1]
sizes = [kept_count, removed_count]
labels_pie = [f'Mantidas\n({kept_count})\nCAUSA only', f'Removidas\n({removed_count})\nEFEITO, OHE\nexpl., NULL']
colors_pie = ['#43A047', '#E53935']
explode = (0.05, 0.1)
axes[1].pie(sizes, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
           explode=explode, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('2. Features Removidas vs Mantidas')

# [3] Heatmap de Correlação Pós-Seleção (sample)
# Reconstruir dataframe com feature nomes para análise
if X_scaled.shape[1] <= 50:  # Se temos poucos features, mostrar todos
    X_scaled_df = pd.DataFrame(X_scaled)
    corr_post = X_scaled_df.corr()
    # Mostrar apenas algumas features para legibilidade
    sns.heatmap(corr_post.iloc[:15, :15], annot=False, cmap='coolwarm', center=0,
               ax=axes[2], cbar_kws={'label': 'Correlação'}, vmin=-1, vmax=1)
    axes[2].set_title('3. Correlação Pós-Seleção (primeiras 15 features)')
else:
    X_scaled_df = pd.DataFrame(X_scaled)
    corr_post = X_scaled_df.corr()
    sns.heatmap(corr_post.iloc[:20, :20], annot=False, cmap='coolwarm', center=0,
               ax=axes[2], cbar_kws={'label': 'Correlação'}, vmin=-1, vmax=1)
    axes[2].set_title('3. Correlação Pós-Seleção (primeiras 20 features)')

plt.tight_layout()
feat_selection_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'feature_selection_impact.png')
fig.savefig(feat_selection_path, dpi=100, bbox_inches='tight')
plt.close()

print(f'\n✅ Feature Selection Impact salva em: {feat_selection_path}')



📊 VISUALIZAÇÃO: Feature Selection Impact

✅ Feature Selection Impact salva em: D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans/feature_selection_impact.png


In [12]:
# ===== [4c] PCA: REDUÇÃO DIMENSIONAL (95% variância) =====
from sklearn.decomposition import PCA

pca = PCA(n_components=3, random_state=42)
X_pca = pca.fit_transform(X_scaled)
n_components = pca.n_components_
explained_var = sum(pca.explained_variance_ratio_)

print(f'\n📉 PCA: REDUÇÃO DIMENSIONAL')
print(f'   Features antes: {X_scaled.shape[1]}')
print(f'   Componentes após: {n_components}')
print(f'   Variância explicada: {explained_var*100:.2f}%')
print(f'   Shape pós-PCA: {X_pca.shape}')


📉 PCA: REDUÇÃO DIMENSIONAL
   Features antes: 17
   Componentes após: 3
   Variância explicada: 49.32%
   Shape pós-PCA: (121751, 3)


In [13]:
# ===== [5] K-MEANS TRAINING (no espaço PCA) =====
k = 4

# Verificar se PCA foi executado (célula 4c)
if 'X_pca' not in locals():
    print('⚠️ Aviso: X_pca não definido. Usando X_scaled diretamente.')
    X_train = X_scaled
    n_components = X_scaled.shape[1]
else:
    X_train = X_pca
    if 'n_components' not in locals():
        n_components = X_pca.shape[1]

model = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
print(f'\n🤖 Treinando K-Means com k={k} (espaço com {n_components} dimensões)...')

try:
    labels = model.fit_predict(X_train)

    sil_score = silhouette_score(X_train, labels)
    db_score = davies_bouldin_score(X_train, labels)

    print(f'\n📊 Métricas de Clustering:')
    print(f'   Silhouette Score: {sil_score:.4f}')
    print(f'   Davies-Bouldin Index: {db_score:.4f}')
    print(f'   Dataset: {len(labels):,} amostras')
    print(f'   ✅ Modelo KMeans treinado com sucesso')
except Exception as e:
    print(f'❌ Erro ao treinar KMeans: {str(e)}')
    print(f'   Verifique se as células anteriores foram executadas corretamente')
    raise


🤖 Treinando K-Means com k=4 (espaço com 3 dimensões)...

📊 Métricas de Clustering:
   Silhouette Score: 0.4551
   Davies-Bouldin Index: 0.7972
   Dataset: 121,751 amostras
   ✅ Modelo KMeans treinado com sucesso


In [14]:
# ===== [6a] ATRIBUIÇÃO DE CLUSTERS (limpos + outliers) =====
# Adicionar clusters ao dataset limpo
df_clean['cluster_pred'] = labels
cluster_names = {0: 'A', 1: 'B', 2: 'C', 3: 'D'}
df_clean['cluster_label'] = df_clean['cluster_pred'].map(cluster_names)

print(f'\n✅ Clusters atribuídos ao dataset limpo ({len(df_clean):,} registros)')

# ===== PREDIZER CLUSTERS PARA OS OUTLIERS REMOVIDOS =====
df_outliers = df[~outlier_mask].copy() if 'outlier_mask' in locals() else pd.DataFrame()

if len(df_outliers) > 0:
    print(f'\n🔄 Atribuindo clusters aos {len(df_outliers):,} incidentes removidos como outliers...')

    try:
        # Define as colunas categoricas que foram usadas para OneHotEncoding
        low_cardinality_cols = ['categoria', 'turno_abertura']
        high_cardinality_cols = ['grupo_designado', 'subcategoria', 'produto']
        temporal_cyclic_cols = ['hora_abertura', 'dia_semana_num', 'mes_abertura']

        # Preparar features dos outliers (mesmo processo que df_clean)
        X_outliers = df_outliers[cause_cols_valid].copy() if 'cause_cols_valid' in locals() else df_outliers.copy()

        # 1. Cyclical encoding para features temporais
        for col, period in zip(temporal_cyclic_cols, [24, 7, 12]):
            if col in X_outliers.columns:
                X_outliers[f'{col}_sin'] = np.sin(2 * np.pi * X_outliers[col] / period)
                X_outliers[f'{col}_cos'] = np.cos(2 * np.pi * X_outliers[col] / period)

        # 2. Frequency encoding para alta cardinalidade (calculado a partir de df_clean)
        for col in high_cardinality_cols:
            if col in X_outliers.columns:
                # Usar frequências do dataset limpo
                freq_map = df_clean[col].value_counts(normalize=True).to_dict()
                X_outliers[col] = X_outliers[col].map(freq_map).fillna(0)

        # 3. One-Hot Encoding com mesmo encoder
        if 'encoder' in locals() and low_cardinality_cols:
            for col in low_cardinality_cols:
                if col in X_outliers.columns:
                    X_outliers[col] = X_outliers[col].fillna('unknown').astype(str)

            X_outliers_cat = encoder.transform(X_outliers[low_cardinality_cols])
            X_outliers_cat_df = pd.DataFrame(X_outliers_cat, columns=encoder.get_feature_names_out(low_cardinality_cols))

            # Selecionar apenas as colunas numericas que foram usadas
            numeric_cols_use = [c for c in X_outliers.columns if c not in low_cardinality_cols and
                               X_outliers[c].dtype in ['int64', 'float64']]
            X_outliers_num = X_outliers[numeric_cols_use].reset_index(drop=True)
            X_outliers_proc = pd.concat([X_outliers_num, X_outliers_cat_df.reset_index(drop=True)], axis=1)
        else:
            X_outliers_proc = X_outliers

        # 4. Escalar com mesmo scaler
        if 'scaler' in locals():
            X_outliers_scaled = scaler.transform(X_outliers_proc.fillna(0))
        else:
            X_outliers_scaled = X_outliers_proc.fillna(0)

        # 5. Aplicar PCA com mesmo transformer
        if 'pca' in locals() and 'X_pca' in locals():
            X_outliers_pca = pca.transform(X_outliers_scaled)
            labels_outliers = model.predict(X_outliers_pca)
        else:
            labels_outliers = model.predict(X_outliers_scaled)

        df_outliers['cluster_pred'] = labels_outliers
        df_outliers['cluster_label'] = df_outliers['cluster_pred'].map(cluster_names)

        print(f'   ✅ Clusters preditos para outliers')
    except Exception as e:
        print(f'   ⚠️ Erro ao predizer clusters para outliers: {str(e)}')
        print(f'   Continuando apenas com dataset limpo')
        df_outliers = pd.DataFrame()
else:
    print('   ℹ️ Nenhum outlier removido, pulando etapa')

# ===== COMBINAR TODOS OS INCIDENTES =====
if len(df_outliers) > 0:
    df_final = pd.concat([df_clean, df_outliers], ignore_index=False).sort_index()
else:
    df_final = df_clean.copy()

print(f'\n📋 Distribuição Final de Clusters:')
cluster_dist = df_final['cluster_label'].value_counts().sort_index()
for label in ['A', 'B', 'C', 'D']:
    count = cluster_dist.get(label, 0)
    pct = count / len(df_final) * 100 if count > 0 else 0
    print(f'   Cluster {label}: {count:>8,} incidentes ({pct:>5.2f}%)')

print(f'\n✅ Dataset Final:')
print(f'   Total: {len(df_final):,} incidentes (100.00%)')
print(f'   Todos os incidentes receberam cluster label')


✅ Clusters atribuídos ao dataset limpo (121,751 registros)

🔄 Atribuindo clusters aos 60 incidentes removidos como outliers...
   ⚠️ Erro ao predizer clusters para outliers: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- dia_semana_num
- hora_abertura
- mes_abertura

   Continuando apenas com dataset limpo

📋 Distribuição Final de Clusters:
   Cluster A:   22,571 incidentes (18.54%)
   Cluster B:   31,821 incidentes (26.14%)
   Cluster C:   38,334 incidentes (31.49%)
   Cluster D:   29,025 incidentes (23.84%)

✅ Dataset Final:
   Total: 121,751 incidentes (100.00%)
   Todos os incidentes receberam cluster label


In [15]:
# ===== [6b] CLUSTER PROFILING: INTERPRETAR COM FEATURES EFEITO =====
print('[INFO] CLUSTER PROFILING: Interpretacao dos Clusters')

# Features de EFEITO para interpretar clusters
effect_cols = [
    'duracao_horas', 'horas_ate_resolucao', 'foi_resolvido', 'excedeu_tempo_esperado',
    'fechado_sem_tecnico', 'target_risco_sla', 'score_risco_operacional'
]

# Usar colunas de EFEITO que já existem em df_final (herdadas de df_clean)
effect_cols_available = [c for c in effect_cols if c in df_final.columns]
df_with_effect = df_final.copy()

# Criar visualizacao com 4 graficos
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Cluster Profiling: Interpretacao com Features EFEITO', fontsize=14, fontweight='bold')

colors_map = {'A': '#E53935', 'B': '#1E88E5', 'C': '#43A047', 'D': '#8E24AA'}

# [1] Tamanho dos clusters
cluster_sizes = df_final['cluster_label'].value_counts().sort_index()
bars = axes[0, 0].bar(cluster_sizes.index, cluster_sizes.values,
                       color=[colors_map[c] for c in cluster_sizes.index],
                       edgecolor='black', linewidth=2)
axes[0, 0].set_ylabel('Numero de Incidentes')
axes[0, 0].set_title('1. Distribuicao de Clusters')
axes[0, 0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, cluster_sizes.values):
    pct = val/len(df_final)*100
    axes[0, 0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
                   f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

# [2] Tempo medio de resolucao por cluster
if 'horas_ate_resolucao' in df_with_effect.columns:
    df_with_effect.boxplot(column='horas_ate_resolucao', by='cluster_label', ax=axes[0, 1])
    axes[0, 1].set_xlabel('Cluster')
    axes[0, 1].set_ylabel('Horas ate Resolucao')
    axes[0, 1].set_title('2. Tempo de Resolucao por Cluster')
    axes[0, 1].get_figure().suptitle('')  # Remove o titulo automatico

# [3] Perfil de metricas EFEITO por cluster (heatmap normalizado)
numeric_effect = [c for c in effect_cols_available if c in df_with_effect.columns and df_with_effect[c].dtype in ['int64', 'float64']]
if numeric_effect:
    profile = df_with_effect.groupby('cluster_label')[numeric_effect].mean()
    profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)
    sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='RdYlGn_r', ax=axes[1, 0],
               cbar_kws={'label': 'Score Normalizado'})
    axes[1, 0].set_title('3. Perfil de Metricas EFEITO (normalizado)')
    axes[1, 0].set_ylabel('Cluster')

# [4] Taxa de resolucao por cluster
if 'foi_resolvido' in df_with_effect.columns:
    resolution_rate = df_with_effect.groupby('cluster_label')['foi_resolvido'].apply(lambda x: (x.sum()/len(x)*100))
    bars = axes[1, 1].bar(resolution_rate.index, resolution_rate.values,
                          color=[colors_map[c] for c in resolution_rate.index],
                          edgecolor='black', linewidth=2)
    axes[1, 1].set_ylabel('Taxa de Resolucao (%)')
    axes[1, 1].set_title('4. Taxa de Resolucao por Cluster')
    axes[1, 1].set_ylim([0, 105])
    axes[1, 1].grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, resolution_rate.values):
        axes[1, 1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                       f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
cluster_profile_path = os.path.join(Path(r'D:\Projetos\AWS_Portifolio\Projeto aws\data\ml\kmeans'), 'cluster_profile_efeito.png')
fig.savefig(cluster_profile_path, dpi=100, bbox_inches='tight')
plt.close()

print('[OK] Cluster Profiling salvo')

# Mostrar resumo interpretativo
print('[INFO] INTERPRETACAO DOS CLUSTERS:')
for label in sorted(df_final['cluster_label'].unique()):
    mask = df_final['cluster_label'] == label
    count = mask.sum()
    pct = count/len(df_final)*100
    print(f'  Cluster {label}: {count:,} incidentes ({pct:.1f}%)')

    if 'horas_ate_resolucao' in df_with_effect.columns:
        avg_duration = df_with_effect[mask]['horas_ate_resolucao'].mean()
        print(f'           Tempo medio: {avg_duration:.1f} horas')

    if 'excedeu_tempo_esperado' in df_with_effect.columns:
        sla_violation = df_with_effect[mask]['excedeu_tempo_esperado'].mean() * 100
        print(f'           Taxa SLA violado: {sla_violation:.1f}%')

[INFO] CLUSTER PROFILING: Interpretacao dos Clusters
[OK] Cluster Profiling salvo
[INFO] INTERPRETACAO DOS CLUSTERS:
  Cluster A: 22,571 incidentes (18.5%)
           Tempo medio: -988993.2 horas
           Taxa SLA violado: 13.1%
  Cluster B: 31,821 incidentes (26.1%)
           Tempo medio: -148733.6 horas
           Taxa SLA violado: 35.9%
  Cluster C: 38,334 incidentes (31.5%)
           Tempo medio: -913350.3 horas
           Taxa SLA violado: 13.7%
  Cluster D: 29,025 incidentes (23.8%)
           Tempo medio: -986550.8 horas
           Taxa SLA violado: 15.0%


In [16]:
# Garante que a coluna de clusters seja atribuída
df_final['cluster_id'] = df_final['cluster_label']

# Exporta o dataset preenchido para formato Parquet
df_final.to_parquet('base_incidentes_clusterizada.parquet', index=False)

print("✅ Arquivo 'base_incidentes_clusterizada.parquet' gerado com sucesso!")

✅ Arquivo 'base_incidentes_clusterizada.parquet' gerado com sucesso!
